In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key=os.getenv("PINECONE_API_KEY")

In [4]:
from pinecone import Pinecone,ServerlessSpec
index_name="hybrid-search-langchain-pinecone"
pc=Pinecone(api_key=api_key)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='dotproduct',
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )

In [5]:
index=pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-a08ipu2.svc.aped-4627-b74a.pinecone.io')

In [6]:
##vector embedding and sparse metrix--- vector embedding for similarity search and sparse metrix for exact search
from langchain_huggingface import HuggingFaceEmbeddings
os.environ["HF_TOKEN"]=os.getenv("HUGGINGFACE_TOKEN")
embeddings=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 635.06it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [9]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder=BM25Encoder().default() ##by default it uses TF-IDF method for sparse metrix
bm25_encoder

In [10]:
sentences=[
    "in 2023, i visited paris",
    "in 2022, i visited new york",
    "in 2021, i visited new orleans"
]
bm25_encoder.fit(sentences)

100%|██████████| 3/3 [00:00<00:00, 10.72it/s]


In [12]:
#storing encoder
bm25_encoder.dump("bm25_values.json")
#loading encoder
bm25_encoder=BM25Encoder().load("bm25_values.json")

In [23]:
dense_vectors=embeddings.embed_documents(sentences)

In [22]:
sparse_vectors=bm25_encoder.encode_documents(sentences)

In [26]:
import uuid
vectors = []

for text, dense, sparse in zip(
    sentences,
    dense_vectors,
    sparse_vectors
):

    vectors.append({
        "id": str(uuid.uuid4()),
        "values": dense,
        "sparse_values": sparse,
        "metadata": {
            "text": text
        }
    })

index.upsert(vectors=vectors)

UpsertResponse(upserted_count=3)

In [27]:
query = "Where did I travel in 2023?"
dense_query = embeddings.embed_query(query)
sparse_query = bm25_encoder.encode_queries(query)
results = index.query(
    vector=dense_query,
    sparse_vector=sparse_query,
    top_k=3,
    include_metadata=True
)

In [28]:
for match in results["matches"]:
    print(match["score"])
    print(match["metadata"]["text"])

0.969692111
in 2023, i visited paris
0.710687637
in 2022, i visited new york
0.577187538
in 2021, i visited new orleans
